# 06 - Ingestão via Streaming -> Medalhão S3 (ano 2026)

Estende o pipeline batch (Bronze/Silver/Gold) para o **ano corrente 2026** via Structured Streaming, escrevendo direto no **Medalhão real no S3**.

**Fluxo (fiel aos notebooks 03/04/05):**
- Simula **entradas cruas** de 2026 no schema de `TS_MUNICIPIO`.
- Cada micro-lote aplica o **SELECT literal da Silver (04)** e, via `foreachBatch`, o **SQL literal da Gold (05)** em modo batch.
- Overwrite dinâmico da partição `ano=2026` -> não encosta em 2023-2025 (donos: batch).
- A **meta 2026 já existe** em `silver/meta_alfabetizacao_municipio` (col. `META_FINAL_2026`, gravada pelo batch). NÃO há bootstrap: reutilizamos a meta do lake.
- Consequência: `percentual_participacao`/`faixa_participacao` **NULL** em 2026; `meta`/`distancia_meta`/`atingiu_meta`/`categoria_desempenho` **preenchidos**.


## 1. Ambiente + credenciais AWS (.env)

In [ ]:
import os
import sys
import shutil
import threading
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

_env = find_dotenv()
# Configuração do ambiente
PROJECT_ROOT = Path(_env).parent if _env else Path.cwd()

# Configuração Spark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Credenciais
AWS_ACCESS_KEY_ID     = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN     = os.getenv("AWS_SESSION_TOKEN")
S3_BUCKET             = os.getenv("S3_BUCKET")


if not all([
    AWS_ACCESS_KEY_ID, 
    AWS_SECRET_ACCESS_KEY, 
    AWS_SESSION_TOKEN, 
    S3_BUCKET,
]):
    raise EnvironmentError(
        "Falha ao carregar credenciais AWS ou S3_BUCKET do arquivo .env"
    )

print("Projeto:", PROJECT_ROOT, "| Bucket:", S3_BUCKET) # remover em produção


# ============================================================
# SESSÃO SPARK
# ============================================================
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("transformar-e-armazenar-S3")
    .master("local[*]") # remover quando for implementar a computação em nuvem
    # Configuração de Pacotes e S3A
    .config("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,""com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.session.token", AWS_SESSION_TOKEN)
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .getOrCreate()
)

## 2. SparkSession COM S3A ativo

Corrige o `ClassNotFoundException: S3AFileSystem`: os pacotes `hadoop-aws` + `aws-java-sdk-bundle` precisam estar carregados. Se já houver uma sessão antiga (sem os jars), encerramos antes — `getOrCreate()` reaproveitaria a antiga.

In [ ]:
from pyspark.sql import SparkSession

# encerra sessão anterior sem os jars S3A (evita reaproveitamento pelo getOrCreate)
# acho que podemos remover no Glue job, já que a sessão é criada uma única vez.
try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("streaming-medalhao-2026")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    )
    .config(
        "spark.hadoop.fs.s3a.impl", 
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.session.token", AWS_SESSION_TOKEN)
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

print(f"Sessao Spark {spark.version} criada (S3A ativo).") # logger.info

## 3. Caminhos no S3, grão da chave e schema CRU dos eventos 2026

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, 
    StructField, 
    IntegerType, 
    StringType, 
    DoubleType
)

ANO_STREAM = 2026

# Camadas do Data Lake
LAKE_BASE   = f"s3a://{S3_BUCKET}"

SILVER_BASE = f"{LAKE_BASE}/silver"
GOLD_BASE   = f"{LAKE_BASE}/gold"

META_SILVER_PATH = f"{SILVER_BASE}/meta_alfabetizacao_municipio"

# --- Streaming: entrada local (produtor) + checkpoint NO S3 ---
STREAM_DIR = PROJECT_ROOT / "data_lake" / "streaming"
INPUT_DIR  = STREAM_DIR / "stream_input_2026"

CKPT       = f"{LAKE_BASE}/_checkpoints/medalhao_2026"

TRIGGER    = "5 seconds"

# ------------------------------------------------------------------------------
# Chaves de negócio
# ------------------------------------------------------------------------------

SILVER_KEY = [
    "ano", 
    "id_municipio", 
    "serie", 
    "rede",
]

# ------------------------------------------------------------------------------
# Schema dos eventos de entrada
# ------------------------------------------------------------------------------

schema_evento = StructType([
    StructField("NU_ANO_AVALIACAO",      IntegerType()),
    StructField("CO_UF",                 IntegerType()),
    StructField("SG_UF",                 StringType()),
    StructField("CO_MUNICIPIO",          StringType()),
    StructField("NO_MUNICIPIO",          StringType()),
    StructField("TP_SERIE",              IntegerType()),
    StructField("ID_TIPO_REDE",          IntegerType()),
    StructField("PC_ALUNO_ALFABETIZADO", DoubleType()),
    StructField("VL_MEDIA_LP",           DoubleType()),
])

print("SILVER_BASE:", SILVER_BASE) # remover em produção
print("GOLD_BASE  :", GOLD_BASE)   # remover em produção
print("CKPT       :", CKPT)        # remover em produção

## 4. Verificação da meta no lake

A meta 2026 já foi gravada pelo batch 04. Aqui só validamos que a tabela existe — usando o `FileSystem` do Hadoop, que distingue "não existe" de "erro de acesso" (não mascara falha de S3 como o try/except largo fazia).

In [ ]:
def _existe_no_lake(path):
    jvm  = spark._jvm
    hconf = spark._jsc.hadoopConfiguration()
    
    p  = jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(hconf)
    
    return fs.exists(p)

if not _existe_no_lake(META_SILVER_PATH):
    raise FileNotFoundError(
        f"Tabela Silver não encontrada: {META_SILVER_PATH}. "
        "Execute o notebook 04 antes da ingestão streaming."
    )

## 5. Simulação das entradas CRUAS de 2026 (schema TS_MUNICIPIO)

Baseada nos municípios reais de 2025. Inclui `ID_TIPO_REDE=3` (Municipal), pois a Gold filtra `rede=3`.

In [ ]:
import random

# ------------------------------------------------------------------------------
# Catálogo de municípios da Silver (gerada no notebook 04)
# ------------------------------------------------------------------------------

catalogo = (
    spark.read
         .parquet(f"{SILVER_BASE}/municipio")
         .filter(F.col("ano") == 2025)
         .select(
             "id_uf",
             "sigla_uf",
             "id_municipio",
             "nome_municipio",
             "serie"
         )
         .dropDuplicates(["id_municipio", "serie"])
         .collect()
)

# ------------------------------------------------------------------------------
# Configuração da simulação
# ------------------------------------------------------------------------------

N_EVENTOS = 200
REDES = [3]

random.seed(2026)

eventos_2026 = []

for _ in range(N_EVENTOS):

    m = random.choice(catalogo)

    eventos_2026.append({

        "NU_ANO_AVALIACAO": ANO_STREAM,

        "CO_UF": m.id_uf,
        "SG_UF": m.sigla_uf,

        "CO_MUNICIPIO": int(m.id_municipio),
        "NO_MUNICIPIO": m.nome_municipio,

        "TP_SERIE": m.serie,

        "ID_TIPO_REDE": random.choice(REDES),

        "PC_ALUNO_ALFABETIZADO": round(
            random.uniform(45, 95),
            2
        ),

        "VL_MEDIA_LP": round(
            random.uniform(730, 800),
            2
        ),
    })

random.shuffle(eventos_2026)   # UFs/redes intercaladas entre os lotes

print(f"Eventos simulados: {len(eventos_2026)}")  # remover em produção

## 6. Produtor: grava micro-lotes CSV crus na entrada local

In [ ]:
import shutil
import time

# Encerra streams ativos
for q in spark.streams.active:
    q.stop()

# Limpa diretório de entrada    
if INPUT_DIR.exists(): shutil.rmtree(INPUT_DIR)

INPUT_DIR.mkdir(parents=True, exist_ok=True)

COLS = [
    "NU_ANO_AVALIACAO",
    "CO_UF","SG_UF",
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "TP_SERIE",
    "ID_TIPO_REDE",
    "PC_ALUNO_ALFABETIZADO",
    "VL_MEDIA_LP"
]

def escrever_lote(rows, nome):
    linhas = [";".join(COLS)]
    for r in rows:
        linhas.append(";".join(str(r[c]) for c in COLS))
    tmp = INPUT_DIR / f"_tmp_{nome}"
    tmp.write_text("\n".join(linhas), encoding="utf-8")
    os.replace(tmp, INPUT_DIR / nome)

def produzir(eventos, n_por_lote=30, intervalo_s=4):
    for n, i in enumerate(range(0, len(eventos), n_por_lote)):
        escrever_lote(eventos[i:i+n_por_lote], f"lote_{n:03d}.csv")
        print(f"[PRODUTOR] lote_{n:03d}.csv -> {len(eventos[i:i+n_por_lote])} eventos")
        time.sleep(intervalo_s)

## 7. `foreachBatch`: Silver row-wise -> acumula partição 2026 -> Gold batch

In [ ]:
from pyspark.sql import functions as F


# ------------------------------------------------------------------------------
# TRANSFORM SILVER
# ------------------------------------------------------------------------------

def transform_silver(df_raw):

    return (
        df_raw
        .filter(F.col("CO_MUNICIPIO").isNotNull())
        .select(
            F.col("NU_ANO_AVALIACAO").cast("int").alias("ano"),
            F.lpad(
                F.trim(F.col("CO_MUNICIPIO").cast("string")),
                7,
                "0"
            ).alias("id_municipio"),
            F.trim(F.col("NO_MUNICIPIO")).alias("nome_municipio"),
            F.col("CO_UF").cast("int").alias("id_uf"),
            F.col("SG_UF").alias("sigla_uf"),
            F.col("TP_SERIE").cast("int").alias("serie"),
            F.col("ID_TIPO_REDE").cast("int").alias("rede"),
            F.col("PC_ALUNO_ALFABETIZADO")
                .cast("double")
                .alias("taxa_alfabetizacao"),
            F.col("VL_MEDIA_LP")
                .cast("double")
                .alias("media_portugues")
        )
        .withColumn("_silver_processed_at", F.current_timestamp())
        .withColumn("processing_timestamp", F.current_timestamp())
        .withColumn("processing_layer", F.lit("streaming"))
        .withColumn("processing_year", F.lit(ANO_STREAM))
    )


# ------------------------------------------------------------------------------
# TRANSFORM GOLD
# ------------------------------------------------------------------------------

def transform_gold(df_silver_2026):
    # Garanta que a view filtre pela rede 3, exatamente como no notebook 05
    df_silver_2026.filter(F.col("rede") == 3).createOrReplaceTempView("silver_municipio")

    (
        spark.read
             .parquet(META_SILVER_PATH)
             .createOrReplaceTempView("silver_meta_municipio")
    )

    return spark.sql("""
    WITH
    -- (1) META -> LONG + COALESCE latest-non-null: 1 linha por (municipio, ano-alvo)
    meta_long_raw AS (
        SELECT ano AS ano_pub, id_municipio,
               stack(7,
                 2024, meta_alfabetizacao_2024, 2025, meta_alfabetizacao_2025,
                 2026, meta_alfabetizacao_2026, 2027, meta_alfabetizacao_2027,
                 2028, meta_alfabetizacao_2028, 2029, meta_alfabetizacao_2029,
                 2030, meta_alfabetizacao_2030
               ) AS (ano, meta)
        FROM silver_meta_municipio
    ),
    meta_long AS (
        SELECT id_municipio, ano, meta FROM (
            SELECT *, ROW_NUMBER() OVER (
                PARTITION BY id_municipio, ano
                ORDER BY (meta IS NOT NULL) DESC, ano_pub DESC
            ) rn FROM meta_long_raw
        ) WHERE rn = 1 AND meta IS NOT NULL
    ),
    -- (2) RESULTADO: já filtrado pela rede 3 na view silver_municipio
    res AS (
        SELECT ano, id_municipio, nome_municipio, id_uf, sigla_uf,
               taxa_alfabetizacao, media_portugues
        FROM silver_municipio
    ),
    -- (3) PARTICIPAÇÃO
    part AS (
        SELECT ano, id_municipio, percentual_participacao
        FROM silver_meta_municipio
    )
    SELECT
        r.ano,
        r.id_municipio,
        r.nome_municipio,
        r.id_uf,
        r.sigla_uf,
        3 AS rede, 
        r.taxa_alfabetizacao,
        r.media_portugues,
        p.percentual_participacao,
        ml.meta,
        ROUND(r.taxa_alfabetizacao - ml.meta, 2)                        AS distancia_meta,
        CASE WHEN ml.meta IS NULL THEN NULL
             ELSE (r.taxa_alfabetizacao - ml.meta) >= 0 END            AS atingiu_meta,
        CASE WHEN ml.meta IS NULL                      THEN NULL
             WHEN r.taxa_alfabetizacao - ml.meta >= 5  THEN 'Muito acima'
             WHEN r.taxa_alfabetizacao - ml.meta >= 0  THEN 'Acima'
             WHEN r.taxa_alfabetizacao - ml.meta >= -5 THEN 'Próximo'
             ELSE 'Muito abaixo' END                                   AS categoria_desempenho,
        CASE WHEN p.percentual_participacao IS NULL THEN NULL
             WHEN p.percentual_participacao >= 95   THEN 'Alta'
             WHEN p.percentual_participacao >= 80   THEN 'Média'
             ELSE 'Baixa' END                                          AS faixa_participacao
    FROM res r
    LEFT JOIN part      p  ON r.ano = p.ano AND r.id_municipio = p.id_municipio
    LEFT JOIN meta_long ml ON r.ano = ml.ano AND r.id_municipio = ml.id_municipio
    """)


# ------------------------------------------------------------------------------
# PROCESSAMENTO DO MICRO-BATCH
# ------------------------------------------------------------------------------

def process_micro_batch(df_raw, epoch_id):

    if df_raw.rdd.isEmpty():
        return

    df_silver = transform_silver(df_raw)

    (
        df_silver.write
            .mode("overwrite")
            .partitionBy("ano", "rede")
            .parquet(f"{SILVER_BASE}/municipio")
    )

    df_silver_2026 = (
        spark.read
             .parquet(f"{SILVER_BASE}/municipio")
             .filter(F.col("ano") == ANO_STREAM)
    )

    df_gold = transform_gold(df_silver_2026)

    (
        df_gold.write
            .mode("overwrite")
            .partitionBy("ano")
            .parquet(f"{GOLD_BASE}/indicadores_municipio")
    )

## 8. Inicia o stream + produtor

In [ ]:
df_stream = (
    spark.readStream
         .schema(schema_evento)
         .option("header", True)
         .option("sep", ";")
         .csv(str(INPUT_DIR))
)

query = (
    df_stream.writeStream
        .foreachBatch(process_micro_batch)
        .option("checkpointLocation", CKPT)
        .outputMode("append")
        .trigger(processingTime=TRIGGER)
        .start()
)

produtor = threading.Thread(
    target=produzir,
    args=(eventos_2026,),
    daemon=False
)

produtor.start()
produtor.join()

query.processAllAvailable()
query.stop()

## 9. Verificação: a Gold 2026 aparece junto de 2023-2025

In [ ]:
# ------------------------------------------------------------------------------
# Validação da camada Gold
# ------------------------------------------------------------------------------

gold = spark.read.parquet(
    f"{GOLD_BASE}/indicadores_municipio"
)

gold.groupBy("ano").count().orderBy("ano").show()

gold.filter(
    F.col("ano") == ANO_STREAM
).show(truncate=False)

In [ ]:
for q in spark.streams.active:
    q.stop()


### Validação


In [ ]:
# ------------------------------------------------------------------------------
# Validação final da camada Gold
# ------------------------------------------------------------------------------

gold = spark.read.parquet(f"{GOLD_BASE}/indicadores_municipio")

print("Registros por ano:")

(
    gold
    .groupBy("ano")
    .count()
    .orderBy("ano")
    .show()
)

print("Amostra dos indicadores de 2026:")

(
    gold
    .filter(F.col("ano") == ANO_STREAM)
    .select(
        "ano",
        "id_municipio",
        "sigla_uf",
        "taxa_alfabetizacao",
        "meta",
        "distancia_meta",
        "atingiu_meta",
        "categoria_desempenho"
    )
    .show(10, truncate=False)
)